# SAC Irrigation Training — v2.5.0

## Key changes from v2.4.2
| # | Change | Why |
|---|--------|-----|
| 1 | `c_term = 0` | Terminal bonus was 300× larger than per-step rewards → Bellman bootstrapping explosion |
| 2 | `ent_coef = 0.05` (hardcoded, not `'auto'`) | Auto-tuner spiked ent_coef to 1.28 at step ~100k, injecting random exploration noise that shattered the Critic |
| 3 | `max_grad_norm = 1.0` | Breaks the positive feedback: large critic loss → large gradients → larger errors |
| 4 | LR decay: 3e-4 → 5e-5 | Stabilises late-training updates |
| 5 | `alpha5_rl = 0.0` | ΔU penalty suppressed weather-responsive behaviour; removed from RL reward only |

## Diagnostic checks to run BEFORE submitting the full 500k job
Run the 10k smoke check cell first.  Expected at step 10k:
- `critic_loss < 50` (was 2700 at step 1k, should converge fast)
- Policy completes full 93-day seasons
- No entropy spike in WandB (ent_coef stays constant at 0.05)

**Change `SEED` to 0, 1, 2, 3, 4 and run one Kaggle notebook per seed.**

In [ ]:
# ── Cell 1: Install & clone ──────────────────────────────────────────────────
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-2000:])
        raise RuntimeError(f'Command failed: {cmd}')
    return result.stdout

# Install dependencies
run('pip install stable-baselines3>=2.6.0 gymnasium casadi wandb --quiet')

# Clone repo
import os
if not os.path.exists('/kaggle/working/thesis'):
    run('git clone https://github.com/taratorbati/thesis.git /kaggle/working/thesis')
else:
    run('cd /kaggle/working/thesis && git pull')

os.chdir('/kaggle/working/thesis')
sys.path.insert(0, '/kaggle/working/thesis')
print('Setup complete. CWD:', os.getcwd())

In [ ]:
# ── Cell 2: WandB secret ────────────────────────────────────────────────────
# The API key must be stored in Kaggle → Add-ons → Secrets as 'WANDB_API_KEY'
# (40-character hex string from https://wandb.ai/authorize → Generate new key)
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
    print('WandB API key loaded from Kaggle secrets.')
except Exception as e:
    print(f'Could not load WandB key ({e}); training will continue without WandB.')

In [ ]:
# ── Cell 3: Smoke test + 10k diagnostic run ─────────────────────────────────
# Run this BEFORE the full 500k job to verify v2.5 stability.
# Expected: critic_loss < 50 at step 10k, no ent_coef spike.
import subprocess
result = subprocess.run(
    'python -m pytest tests/test_rl_smoke.py -v',
    shell=True, capture_output=True, text=True
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print('SMOKE TESTS FAILED — do not proceed to full training')
    print(result.stderr[-2000:])
else:
    print('All smoke tests passed.')

In [ ]:
# ── Cell 4: Full 500k training ───────────────────────────────────────────────
# Change SEED for each parallel Kaggle session (0, 1, 2, 3, 4)
SEED = 0   # <--- CHANGE THIS

from src.rl.train import train_sac

model = train_sac(
    seed=SEED,
    output_dir='/kaggle/working/thesis/results/rl',
    wandb_project='sac-irrigation-thesis',
    total_timesteps=500_000,
)
print(f'Training complete for seed {SEED}.')

In [ ]:
# ── Cell 5: Copy results to output ───────────────────────────────────────────
import shutil, os
src = f'/kaggle/working/thesis/results/rl/sac_general_seed{SEED}'
dst = f'/kaggle/working/results_seed{SEED}'
# Exclude the large replay buffer from the output copy
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst, ignore=shutil.ignore_patterns('replay_buffer_latest.pkl'))
print(f'Results copied to {dst} (replay buffer excluded).')
print('Files:', [f for root,_,files in os.walk(dst) for f in files])

In [ ]:
# ── Cell 6: Zip & download ───────────────────────────────────────────────────
import shutil
archive = f'/kaggle/working/sac_seed{SEED}_v25'
shutil.make_archive(archive, 'zip', f'/kaggle/working/results_seed{SEED}')
print(f'Archive ready: {archive}.zip')
print('Download via Kaggle → Output → Files panel.')

In [ ]:
# ── Cell 7: Resume from checkpoint (if session was killed) ───────────────────
# Uncomment and run this cell to resume training from the latest checkpoint.
# You must upload the checkpoint zip back to Kaggle first.

# SEED = 0
# CHECKPOINT_STEPS = 200_000   # match the checkpoint filename
# CHECKPOINT_ZIP = f'/kaggle/input/sac-checkpoint/sac_general_seed{SEED}_{CHECKPOINT_STEPS}_steps.zip'
# BUFFER_PKL     = f'/kaggle/input/sac-checkpoint/replay_buffer_latest.pkl'
#
# from stable_baselines3 import SAC
# from src.rl.networks import CTDESACPolicy
# from src.rl.train import train_sac, _make_lr_schedule, LR_START, LR_END
# from stable_baselines3.common.vec_env import DummyVecEnv
# from src.rl.gym_env import IrrigationEnv
#
# env = DummyVecEnv([lambda: IrrigationEnv(randomize=True)])
# model = SAC.load(CHECKPOINT_ZIP, env=env)
# model.load_replay_buffer(BUFFER_PKL)
# # Re-apply the LR schedule (SB3 does not persist schedules in .zip)
# model.lr_schedule = _make_lr_schedule(LR_START, LR_END)
# remaining = 500_000 - CHECKPOINT_STEPS
# model.learn(total_timesteps=remaining, reset_num_timesteps=False, progress_bar=True)
# model.save(f'/kaggle/working/thesis/results/rl/sac_general_seed{SEED}/sac_general_seed{SEED}_final')
# print('Resume complete.')